# Seasonal Adjustment for Russia — Python Demo

Этот notebook демонстрирует Python-порт функции `sa_ru` + **пакетную обработку** `sa_ru_batch` из репозитория
[NadezhdaYurchenko/Seasonal-Adjustment-for-Russia](https://github.com/NadezhdaYurchenko/Seasonal-Adjustment-for-Russia).

## Содержание
1. Генерация синтетических рядов (3 ряда с разными свойствами)
2. **`sa_ru_batch()`** — пакетная обработка всех рядов за один вызов
3. Индивидуальные настройки для каждого ряда
4. Сравнение R vs Python (R², MAD)
5. Сохранение результатов в Excel

> **Примечание:** R использует X-13ARIMA-SEATS (Census Bureau), Python использует `statsmodels.SARIMAX`.
> Логика центрирования регрессоров и интерфейс идентичны.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from sa_ru_python import sa_ru, sa_ru_batch, sa_ru_batch_to_excel

print('Импорты выполнены успешно.')

## 1. Генерация синтетических рядов

Создаём **3 ряда** с разными характеристиками:
- **IP** (Индекс промышленного производства) — положительный уровень, мультипликативная сезонность, хочется `transform=log`
- **CPI_mom** (Инфляция м/м) — темп прироста может быть отрицательным, аддитивная модель
- **Retail** (Розничные продажи) — с Пасхой и новогодним провалом, extended-режим

Это типичная ситуация, когда **каждый ряд требует своих настроек**.

In [ ]:
np.random.seed(42)
n_months = 120  # 10 лет
dates = pd.date_range('2014-01-01', periods=n_months, freq='MS')
t = np.arange(n_months)

# --- IP: индекс производства (мультипликативная сезонность)
ip_trend    = 100 * np.exp(t * 0.002)  # +~2.4% год
ip_season   = np.array([-0.05, -0.02, 0.01, 0.02, 0.01, 0.03,
                          0.03,  0.04, 0.02, 0.01, -0.005, -0.02])
ip_seas_vec = np.array([ip_season[m % 12] for m in range(n_months)])
ip_values   = ip_trend * (1 + ip_seas_vec) + np.random.normal(0, 0.5, n_months)

# --- CPI_mom: месячная инфляция (аддитивная)
cpi_trend   = 0.3 + t * (-0.001)  # постепенное снижение инфляции
cpi_season  = np.array([-0.3, 0.2, 0.1, 0.1, 0.0, 0.05,
                          0.05, 0.0, -0.1, -0.05, 0.1, 0.4])
cpi_seas_v  = np.array([cpi_season[m % 12] for m in range(n_months)])
cpi_values  = cpi_trend + cpi_seas_v + np.random.normal(0, 0.1, n_months)

# --- Retail: розничные продажи (с Пасхой в апреле-мае)
ret_trend   = 80 + t * 0.5
ret_season  = np.array([-8, -4, 2, 4, 5, 3, 2, 3, 2, 1, 5, 12])  # декабрь-пик
ret_seas_v  = np.array([ret_season[m % 12] for m in range(n_months)])
ret_values  = ret_trend + ret_seas_v + np.random.normal(0, 1.5, n_months)

# --- Собираем wide DataFrame
wide_df = pd.DataFrame({
    'date':    dates,
    'IP':      ip_values,
    'CPI_mom': cpi_values,
    'Retail':  ret_values,
})

print('Wide DataFrame (первые 4 строки):')
print(wide_df.head(4).to_string(index=False))
print(f'\nРазмер: {wide_df.shape[0]} строк × {wide_df.shape[1]} колонок')

In [ ]:
# Визуализация исходных рядов
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, col, title, color in zip(
    axes,
    ['IP', 'CPI_mom', 'Retail'],
    ['IP (Индекс промпроизводства)', 'CPI м/м (Инфляция)', 'Retail (Розничные продажи)'],
    ['#2563eb', '#dc2626', '#16a34a']
):
    ax.plot(dates, wide_df[col], color=color, linewidth=1.5)
    ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.suptitle('Исходные ряды (wide формат)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('batch_input_series.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Пакетная обработка `sa_ru_batch()`

**Ключевая фича:** каждый ряд получает свои настройки через `series_configs`.

```python
series_configs = {
    'IP':      {'transform_function': 'log'},   # логарифм для уровневого ряда
    'CPI_mom': {'calendar_mode': 'none'},        # без рабочих дней (темп прироста)
    'Retail':  {'calendar_mode': 'extended', 'include_easter': True},
}
```

In [ ]:
CALENDAR_FILE = 'russia_calendar.xlsx'

import os
if not os.path.exists(CALENDAR_FILE):
    print(f'⚠️  Файл {CALENDAR_FILE} не найден.')
    print('Скачайте russia_calendar.xlsx из репозитория и положите рядом с notebook.')
else:
    print(f'✓ Файл календаря найден: {CALENDAR_FILE}')

In [ ]:
%%time
batch_results = sa_ru_batch(
    df            = wide_df,
    calendar_file = CALENDAR_FILE,
    format        = 'wide',         # wide-формат: дата + колонки рядов
    date_col      = 'date',

    # Настройки по умолчанию для всех рядов
    default_config = {
        'calendar_mode':      'basic',
        'include_easter':     True,
        'transform_function': 'none',
        'forecast_months':    12,
    },

    # Индивидуальные настройки для конкретных рядов
    series_configs = {
        'IP':      {'transform_function': 'log'},      # мультипликативная модель
        'CPI_mom': {'calendar_mode': 'none'},           # темп роста — без рабочих дней
        'Retail':  {'calendar_mode': 'extended',
                    'include_easter': True},             # расширенный + Пасха
    },

    fail_on_error = False,   # продолжать если один ряд упал
    verbose       = True,
)

In [ ]:
# Сводная таблица
print('=== Сводка по всем рядам ===')
print(batch_results['summary'].to_string(index=False))

In [ ]:
# Объединённый long DataFrame — удобно для дальнейшего анализа
combined = batch_results['combined']
print('Combined (long) DataFrame:')
print(combined.head(8).to_string(index=False))
print(f'\nВсего строк: {len(combined)}, рядов: {combined["series_id"].nunique()}')

## 3. Визуализация результатов

In [ ]:
series_list = ['IP', 'CPI_mom', 'Retail']
colors = {'original': '#94a3b8', 'adjusted': '#dc2626', 'trend': '#16a34a'}

fig, axes = plt.subplots(len(series_list), 1, figsize=(14, 4*len(series_list)), sharex=True)

for ax, sname in zip(axes, series_list):
    if sname not in batch_results['results']:
        ax.set_title(f'{sname} — ОШИБКА', color='red')
        continue
    
    data = batch_results['results'][sname]['data']
    row  = batch_results['summary'][batch_results['summary']['series_id'] == sname].iloc[0]
    
    ax.plot(data['date'], data['original'],  color=colors['original'], lw=1.2, alpha=0.7, label='Исходный')
    ax.plot(data['date'], data['adjusted'],  color=colors['adjusted'],  lw=2.0, label='SA (Python)')
    mask = ~data['trend'].isna()
    ax.plot(data['date'][mask], data['trend'][mask], color=colors['trend'], lw=1.5,
            linestyle='--', label='Тренд')
    
    title = (f"{sname}  |  модель={row['chosen_model']}  "
             f"transform={row['transform']}  AIC={row['aic']:.1f}")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.suptitle('Пакетная сезонная корректировка (sa_ru_batch)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('batch_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('График сохранён: batch_results.png')

## 4. Long-формат на вход

`sa_ru_batch()` также принимает **long-формат** — удобно если данные уже в tidy-виде.

In [ ]:
# Конвертируем wide -> long вручную
long_df = wide_df.melt(id_vars='date', var_name='indicator', value_name='value')
print('Long DataFrame:')
print(long_df.head(6).to_string(index=False))

# Запуск sa_ru_batch с long-форматом
batch_from_long = sa_ru_batch(
    df            = long_df,
    calendar_file = CALENDAR_FILE,
    format        = 'long',         # ← long-формат
    date_col      = 'date',
    series_col    = 'indicator',    # ← колонка с именами рядов
    value_col     = 'value',
    default_config = {'calendar_mode': 'basic', 'forecast_months': 12},
    verbose       = False,          # тихий режим
)

print('\nРезультат из long-формата:')
print(batch_from_long['summary'][['series_id', 'status', 'aic']].to_string(index=False))

## 5. Сравнение R и Python (численные метрики)

Имитируем R-результат (истинный тренд + малый шум) и сравниваем с Python.

In [ ]:
true_trends = {'IP': ip_trend, 'CPI_mom': cpi_trend, 'Retail': ret_trend}

def r_squared(a, b):
    mask = ~(np.isnan(a) | np.isnan(b))
    ss_res = np.sum((a[mask] - b[mask])**2)
    ss_tot = np.sum((a[mask] - a[mask].mean())**2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

def mad(a, b):
    mask = ~(np.isnan(a) | np.isnan(b))
    return np.mean(np.abs(a[mask] - b[mask]))

rows = []
np.random.seed(99)
for sname in series_list:
    if sname not in batch_results['results']:
        continue
    data = batch_results['results'][sname]['data']
    adj_py = data['adjusted'].to_numpy()
    trend  = true_trends[sname]
    # Имитируем R-результат
    adj_r = trend + np.random.normal(0, np.std(trend) * 0.003, n_months)
    rows.append({
        'Ряд':                   sname,
        'R² (Python vs тренд)':  f"{r_squared(trend, adj_py):.4f}",
        'R² (R vs тренд)':       f"{r_squared(trend, adj_r):.4f}",
        'R² (Python vs R)':      f"{r_squared(adj_r, adj_py):.4f}",
        'MAD (Python vs R)':     f"{mad(adj_r, adj_py):.4f}",
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

## 6. Сохранение в Excel (один лист на ряд)

In [ ]:
output_path = sa_ru_batch_to_excel(
    batch_result   = batch_results,
    path           = 'sa_results.xlsx',
    include_summary = True,  # лист Summary с общей сводкой
)
print(f'✓ Excel сохранён: {output_path}')
print('Листы: Summary + по одному листу на каждый ряд')

## Итоговое резюме

| Возможность | R (`sa_ru_batch.R`) | Python (`sa_ru_batch`) |
|---|---|---|
| Wide-формат (дата + колонки) | ✅ | ✅ |
| Long-формат (tidy) | ✅ | ✅ |
| Индивид. настройки на ряд | ✅ | ✅ |
| Продолжение при ошибке | ✅ fail_on_error | ✅ fail_on_error |
| Сводная таблица + AIC | ✅ | ✅ |
| Объединённый long-результат | ✅ | ✅ |
| Сохранение в Excel | ✅ openxlsx | ✅ openpyxl |